# PyTorch sequence training: CPU or NVIDIA CUDA

Train CNN, LSTM and GRU models on the v2 windows. `DEVICE = "auto"` selects CUDA when available and CPU otherwise; use `"cuda"` to require an NVIDIA GPU or `"cpu"` for the previous CPU comparison. The pilot uses the saved 2-second `akmal` selection and normalizer. Set `RUN_FULL_TRAINING = True` for all three durations on the complete attempt split.

The timing comparison uses the saved tinygrad/OpenCL pilot. The models share architecture families and input shapes, but not initialized weights or kernels. Timings include loading, validation and checkpoints; CUDA timings synchronize the device. Model quality cannot be attributed to the framework from this pilot.

## Setup

For CPU: `uv sync --inexact --extra analysis --extra detection --extra notebook`. For an NVIDIA GPU on Linux or Windows: `uv sync --inexact --extra analysis --extra cuda --extra notebook`. These extras select different PyTorch wheels and cannot be enabled together. Restart the project kernel after changing environments. The CUDA 13.0 wheel requires a compatible NVIDIA driver and GPU.

The pilot comparison also needs the local saved tinygrad run below. If it is unavailable, set `RUN_PILOT = False` and enable `RUN_FULL_TRAINING` to train directly from the v2 datasets.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import subprocess
import sys
import numpy as np
import polars as pl
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src/fall_detection").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fall_detection.analysis.sequence_training_data import (
    DURATIONS, SegmentStore, fit_normalizer, load_sequence_index,
    split_sequence_attempts,
)

DATA_ROOT = PROJECT_ROOT / "data/processed/amplitude_sequence_v2"
TINYGRAD_RUN = PROJECT_ROOT / "data/processed/tinygrad_segment_pilot/akmal_v2_cpu_comparison"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PYTORCH_RUN = PROJECT_ROOT / "data/processed/pytorch_sequence_pilot/notebook" / RUN_ID
CPU_THREADS = 4
BATCH_SIZE = 32  # Reduce if the CUDA device runs out of memory.
DEVICE = "auto"  # "auto", "cpu", or "cuda"; "cuda" fails if unavailable
PILOT_DURATION = 2
RUN_PILOT = True
RUN_FULL_TRAINING = False
print("PyTorch output:", PYTORCH_RUN, "requested device:", DEVICE)

PyTorch output: /home/khaelano/Projects/radio-fall-detection/data/processed/pytorch_sequence_pilot/notebook/20261111T120955139505Z requested device: auto


In [2]:
if RUN_PILOT:
    selection_path = TINYGRAD_RUN / "profile/window_split.parquet"
    normalizer_path = TINYGRAD_RUN / f"profile/normalizer-{PILOT_DURATION}s.npz"
    if not selection_path.is_file() or not normalizer_path.is_file():
        raise FileNotFoundError("The saved tinygrad comparison pilot is required")
    selection = pl.read_parquet(selection_path).filter(pl.col("duration") == PILOT_DURATION)
    display(selection.group_by("dataset", "split", "target").len().sort("dataset", "split", "target"))
    selection_record = json.loads((TINYGRAD_RUN / "profile/selection.json").read_text())
    assert selection_record["window_split_sha256"] == hashlib.sha256(selection_path.read_bytes()).hexdigest()
    assert set(selection["dataset"]) == {"akmal"}
    print("Pilot windows:", len(selection), "normalizer:", normalizer_path)
else:
    print("Pilot disabled; the saved tinygrad selection is not required")

dataset,split,target,len
str,str,i64,u32
"""akmal""","""test""",0,24
"""akmal""","""test""",1,12
"""akmal""","""train""",0,24
"""akmal""","""train""",1,12
"""akmal""","""validation""",0,24
"""akmal""","""validation""",1,12


Pilot windows: 108 normalizer: /home/khaelano/Projects/radio-fall-detection/data/processed/tinygrad_segment_pilot/akmal_v2_cpu_comparison/profile/normalizer-2s.npz


## Three-epoch PyTorch pilot

A fresh process runs on the requested device and records the actual device, CPU peak RSS, and CUDA peak allocated memory when available. Checkpoints and timings are saved under `PYTORCH_RUN`.

In [3]:
if RUN_PILOT:
    command = [
        sys.executable, "-u", str(PROJECT_ROOT / "tools/run_pytorch_cpu_sequence.py"),
        "--tinygrad-run", str(TINYGRAD_RUN),
        "--output", str(PYTORCH_RUN),
        "--duration", str(PILOT_DURATION),
        "--cpu-threads", str(CPU_THREADS),
        "--device", DEVICE, "--batch-size", str(BATCH_SIZE),
    ]
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError(f"PyTorch pilot failed with exit code {completed.returncode}")
else:
    print("PyTorch pilot disabled; set RUN_PILOT=True to generate this run's metrics")

CNN CPU epoch 1/3: 0.425s
CNN CPU epoch 2/3: 0.391s
CNN CPU epoch 3/3: 0.402s
LSTM CPU epoch 1/3: 0.521s
LSTM CPU epoch 2/3: 0.452s
LSTM CPU epoch 3/3: 0.438s
GRU CPU epoch 1/3: 0.411s
GRU CPU epoch 2/3: 0.398s
GRU CPU epoch 3/3: 0.393s
device cpu GPU None
cnn training_seconds 1.219 peak_rss_bytes 763949056 peak_cuda_allocated_bytes None
lstm training_seconds 1.413 peak_rss_bytes 1072013312 peak_cuda_allocated_bytes None
gru training_seconds 1.202 peak_rss_bytes 1072013312 peak_cuda_allocated_bytes None



## Timing comparison

Both runs use the saved selection, normalizer, epoch count, and batch size. Tinygrad ran on the recorded OpenCL device; PyTorch runs on the device reported below. CUDA steps synchronize before timing is recorded. The runs use different kernels and weights and are not concurrent. First epochs include backend setup; repeat-epoch medians describe only this pilot.

In [4]:
if RUN_PILOT:
    if BATCH_SIZE == 32:
        cpu = json.loads((PYTORCH_RUN / "metrics.json").read_text())
        assert cpu["selection_sha256"] == selection_record["window_split_sha256"]
        actual_device = cpu["runtime"]["device"]
        rows = []
        for kind in ("cnn", "lstm", "gru"):
            gpu = json.loads((TINYGRAD_RUN / "profile" / f"{PILOT_DURATION}s" / kind / "metrics.json").read_text())
            host = cpu["models"][kind]
            assert gpu["counts"] == cpu["counts"]
            assert gpu["config"]["batch_size"] == cpu["config"]["batch_size"]
            assert gpu["config"]["hidden_size"] == cpu["config"]["hidden_size"]
            gpu_steady = float(np.median([e["epoch_seconds"] for e in gpu["history"][1:]]))
            cpu_steady = float(np.median([e["epoch_seconds"] for e in host["history"][1:]]))
            rows.append({
                "model": kind.upper(),
                f"PyTorch {actual_device} total s": round(host["training_seconds"], 3),
                "tinygrad GPU total s": round(gpu["training_seconds"], 3),
                "total tinygrad / PyTorch": round(gpu["training_seconds"] / host["training_seconds"], 2),
                f"PyTorch {actual_device} repeat epoch s": round(cpu_steady, 3),
                "tinygrad repeat epoch s": round(gpu_steady, 3),
                "repeat tinygrad / PyTorch": round(gpu_steady / cpu_steady, 2),
                "PyTorch peak RSS GiB": (round(host["peak_rss_bytes"] / 2**30, 3)
                                           if host["peak_rss_bytes"] is not None else None),
                "PyTorch CUDA peak allocated GiB": (round(host["peak_cuda_allocated_bytes"] / 2**30, 3)
                                                       if host["peak_cuda_allocated_bytes"] is not None else None),
                "tinygrad peak cgroup GiB": round(gpu["worker_memory_peak_bytes"] / 2**30, 3),
            })
        comparison = pl.DataFrame(rows)
        display(comparison)
        print("PyTorch:", cpu["runtime"]["torch"], "device:", actual_device,
              "GPU:", cpu["runtime"]["gpu_name"], "threads:", cpu["runtime"]["cpu_threads"],
              "affinity CPUs:", cpu["runtime"]["affinity_cpus"])
    else:
        print("Pilot comparison skipped: reference uses batch size 32")
else:
    print("Pilot comparison skipped")

model,PyTorch cpu total s,tinygrad GPU total s,total tinygrad / PyTorch,PyTorch cpu repeat epoch s,tinygrad repeat epoch s,repeat tinygrad / PyTorch,PyTorch peak RSS GiB,PyTorch CUDA peak allocated GiB,tinygrad peak cgroup GiB
str,f64,f64,f64,f64,f64,f64,f64,null,f64
"""CNN""",1.219,10.406,8.54,0.397,1.709,4.31,0.711,null,0.758
"""LSTM""",1.413,28.725,20.33,0.445,1.896,4.26,0.998,null,0.957
"""GRU""",1.202,27.011,22.47,0.395,1.636,4.14,0.998,null,1.041


PyTorch: 2.14.1+cpu device: cpu GPU: None threads: 4 affinity CPUs: 4


## Optional complete-data PyTorch training

This section is disabled by default. When enabled, it makes the same seed-42 attempt split across all three v2 datasets, fits a separate training-only normalizer for each duration, then runs CNN/LSTM/GRU training for up to 30 epochs with validation fall-F1 early stopping (patience five). Each duration writes its own checkpoints and metrics. These full runs have not been measured by the pilot table above.

In [5]:
if RUN_FULL_TRAINING:
    full_dir = PYTORCH_RUN / "full"
    full_dir.mkdir(parents=True, exist_ok=False)
    index, sources = load_sequence_index(DATA_ROOT)
    split = split_sequence_attempts(index, seed=42)
    split_path = full_dir / "window_split.parquet"
    split.write_parquet(split_path)
    for duration in DURATIONS:
        normalizer = fit_normalizer(split.filter(pl.col("duration") == duration), SegmentStore())
        statistics_path = full_dir / f"normalizer-{duration}s.npz"
        normalizer.save(statistics_path)
        command = [
            sys.executable, "-u", str(PROJECT_ROOT / "tools/run_pytorch_cpu_sequence.py"),
            "--frame", str(split_path), "--normalizer", str(statistics_path),
            "--output", str(full_dir / f"{duration}s"),
            "--duration", str(duration), "--epochs", "30", "--patience", "5",
            "--cpu-threads", str(CPU_THREADS), "--device", DEVICE, "--batch-size", str(BATCH_SIZE),
        ]
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Complete-data training disabled; set RUN_FULL_TRAINING=True to run it")

Complete-data training disabled; set RUN_FULL_TRAINING=True to run it
